In [4]:
!pip install sqlalchemy psycopg2-binary

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 12.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 22.5 MB/s  0:00:00

   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   ---------------------------------------- 3/3 [sqlalchemy]



In [6]:
import pandas as pd
from sqlalchemy import create_engine

# Cambia estos valores por los tuyos
USER = "postgres.irjzhqrawpjqcicmbwwc"
PASSWORD = "los4fantasticos*"
HOST = "aws-1-us-east-1.pooler.supabase.com"
PORT = "5432"
DBNAME = "postgres"

engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}")

query = """
SELECT *
FROM f1_simulation_laptime_ml;
"""

df = pd.read_sql(query, engine)

print(df.shape)
print(df.head())
print(df.dtypes)

(92514, 26)
   Year                        Event Driver             Team  LapNumber  \
0  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         30   
1  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         31   
2  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         32   
3  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         33   
4  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         34   

   Stint Compound  TyreLife  FreshTyre TrackStatus  ...  SpeedFL  SpeedST  \
0      2     HARD        24       True           1  ...      NaN    297.0   
1      3     HARD         1       True          12  ...    249.0    305.0   
2      3     HARD         2       True           1  ...    250.0    314.0   
3      3     HARD         3       True           1  ...    249.0    348.0   
4      3     HARD         4       True           1  ...    249.0    328.0   

   lap_position    AirTemp   Humidity     Pressure  Rainfall  TrackTemp  \

In [7]:
from sklearn.model_selection import train_test_split

# Variable objetivo
y = df["laptime_seconds"]

# Variables de entrada
X = df.drop(columns=[
    "laptime_seconds",
    "sector1_seconds",
    "sector2_seconds",
    "sector3_seconds"
], errors="ignore")

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())


X shape: (92514, 22)
y shape: (92514,)


,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,...,SpeedFL,SpeedST,lap_position,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,30,2,HARD,24,True,1,...,NaN,297.0,7,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
1,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,31,3,HARD,1,True,12,...,249.0,305.0,11,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
2,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,32,3,HARD,2,True,1,...,250.0,314.0,11,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
3,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,33,3,HARD,3,True,1,...,249.0,348.0,11,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
4,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,34,3,HARD,4,True,1,...,249.0,328.0,10,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [8]:
query = """
SELECT *
FROM f1_simulation_laptime_ml;
"""

df = pd.read_sql(query, engine)

print(df.shape)
print(df["Year"].value_counts().sort_index())
display(df.head())

(142342, 26)
Year
2020    19926
2021    24347
2022    21860
2023    49828
2024    26381
Name: count, dtype: int64


,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,...,SpeedFL,SpeedST,lap_position,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7,True,1,...,250.0,306.0,3,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
1,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,52,3,HARD,22,True,1,...,247.0,261.0,5,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
2,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,50,3,HARD,20,True,1,...,251.0,301.0,5,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
3,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,49,3,HARD,19,True,1,...,254.0,324.0,6,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
4,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,48,3,HARD,18,True,1,...,253.0,300.0,6,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [9]:
from sklearn.model_selection import train_test_split

y = df["laptime_seconds"]

X = df.drop(columns=[
    "laptime_seconds",
    "sector1_seconds",
    "sector2_seconds",
    "sector3_seconds"
], errors="ignore")

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

X shape: (142342, 22)
y shape: (142342,)
Numéricas: ['Year', 'LapNumber', 'Stint', 'TyreLife', 'FreshTyre', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'lap_position', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']
Categóricas: ['Event', 'Driver', 'Team', 'Compound', 'TrackStatus']


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.6600524359829988
RMSE: 2.0346091146944323
R2: 0.9958989712954118


In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RepeatedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# 1. Definir X e y
# =========================================================

y = df["laptime_seconds"]

X_sim = df.drop(columns=[
    "laptime_seconds",
    "sector1_seconds",
    "sector2_seconds",
    "sector3_seconds",
    "SpeedI1",
    "SpeedI2",
    "SpeedFL",
    "SpeedST",
    "lap_position"
], errors="ignore")

print("X_sim shape:", X_sim.shape)
print("y shape:", y.shape)

# =========================================================
# 2. Separar primero el 20% para TEST
# =========================================================

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_sim, y,
    test_size=0.20,
    random_state=42
)

# =========================================================
# 3. Del 80% restante:
#    70% TRAIN
#    30% VAL
# =========================================================

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.30,
    random_state=42
)

print("\nTamaños:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

# =========================================================
# 4. Identificar columnas usando TRAIN
# =========================================================

numeric_features_sim = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_sim = X_train.select_dtypes(include=["object"]).columns.tolist()

print("\nNuméricas simulación:", numeric_features_sim)
print("Categóricas simulación:", categorical_features_sim)

# =========================================================
# 5. Preprocesamiento
# =========================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_sim = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features_sim),
        ("cat", categorical_transformer, categorical_features_sim)
    ]
)

# =========================================================
# 6. Modelo
# =========================================================

model_sim = Pipeline(steps=[
    ("preprocessor", preprocessor_sim),
    ("regressor", RandomForestRegressor(
        n_estimators=30,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

# =========================================================
# 7. Cross validation SOLO en TRAIN
# =========================================================

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

cv_results = cross_validate(
    estimator=model_sim,
    X=X_train,
    y=y_train,
    cv=rkf,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    },
    n_jobs=-1,
    return_train_score=False
)

cv_mae = -cv_results["test_mae"]
cv_rmse = -cv_results["test_rmse"]
cv_r2 = cv_results["test_r2"]

print("\nCross Validation SOLO en TRAIN")
print("MAE CV promedio :", cv_mae.mean())
print("MAE CV std      :", cv_mae.std())
print("RMSE CV promedio:", cv_rmse.mean())
print("RMSE CV std     :", cv_rmse.std())
print("R2 CV promedio  :", cv_r2.mean())
print("R2 CV std       :", cv_r2.std())

# =========================================================
# 8. Entrenar SOLO con TRAIN
# =========================================================

model_sim.fit(X_train, y_train)

# =========================================================
# 9. Evaluar en VALIDACIÓN
# =========================================================

y_val_pred = model_sim.predict(X_val)

mae_val = mean_absolute_error(y_val, y_val_pred)
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2_val = r2_score(y_val, y_val_pred)

print("\nMétricas en VALIDACIÓN")
print("MAE val :", mae_val)
print("RMSE val:", rmse_val)
print("R2 val  :", r2_val)

# =========================================================
# 10. Evaluar al final en TEST
# =========================================================

y_test_pred = model_sim.predict(X_test)

mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

print("\nMétricas en TEST")
print("MAE test :", mae_test)
print("RMSE test:", rmse_test)
print("R2 test  :", r2_test)

X_sim shape: (142342, 17)
y shape: (142342,)

Tamaños:
Train: (79711, 17) (79711,)
Val  : (34162, 17) (34162,)
Test : (28469, 17) (28469,)

Numéricas simulación: ['Year', 'LapNumber', 'Stint', 'TyreLife', 'FreshTyre', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']
Categóricas simulación: ['Event', 'Driver', 'Team', 'Compound', 'TrackStatus']

Cross Validation SOLO en TRAIN
MAE CV promedio : 1.8575983499442432
MAE CV std      : 0.06659356543366124
RMSE CV promedio: 3.951146464664721
RMSE CV std     : 0.17480265404255727
R2 CV promedio  : 0.9708336588748147
R2 CV std       : 0.019162389353853067

Métricas en VALIDACIÓN
MAE val : 1.8358362049853372
RMSE val: 3.7290132053532496
R2 val  : 0.9868611255948634

Métricas en TEST
MAE test : 1.8345436367758814
RMSE test: 3.7463674470506865
R2 test  : 0.9860956215517404


In [20]:
base_row = X_sim[
    (X_sim["Compound"].isin(["SOFT", "MEDIUM", "HARD"])) &
    (X_sim["Rainfall"] == False)
].iloc[[0]].copy()

display(base_row)

,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7,True,1,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [21]:
# Escenario 1: comparar compuestos en seco
compound_options = ["SOFT", "MEDIUM", "HARD"]

scenarios_compound = []

for comp in compound_options:
    row = base_row.copy()
    row["Compound"] = comp
    row["Rainfall"] = False
    
    pred = model_sim.predict(row)[0]
    
    out = row.copy()
    out["scenario_type"] = "compound_dry"
    out["scenario_value"] = comp
    out["predicted_laptime"] = pred
    scenarios_compound.append(out)

sim_compound_dry = pd.concat(scenarios_compound, ignore_index=True)

display(sim_compound_dry[[
    "Driver", "Event", "LapNumber", "Stint",
    "Compound", "TyreLife", "TrackTemp", "Rainfall",
    "predicted_laptime"
]])

,Driver,Event,LapNumber,Stint,Compound,TyreLife,TrackTemp,Rainfall,predicted_laptime
0,BOT,70th Anniversary Grand Prix,39,3,SOFT,7,43.227586,False,95.385941
1,BOT,70th Anniversary Grand Prix,39,3,MEDIUM,7,43.227586,False,91.438138
2,BOT,70th Anniversary Grand Prix,39,3,HARD,7,43.227586,False,91.438138


In [22]:
# Escenario 2: degradación del neumático
tyre_life_options = [1, 5, 10, 15, 20, 25]

scenarios_tyre = []

for tl in tyre_life_options:
    row = base_row.copy()
    row["Compound"] = "HARD"   # dejamos fijo el compuesto base
    row["Rainfall"] = False
    row["TyreLife"] = tl
    
    pred = model_sim.predict(row)[0]
    
    out = row.copy()
    out["scenario_type"] = "tyre_life"
    out["scenario_value"] = tl
    out["predicted_laptime"] = pred
    scenarios_tyre.append(out)

sim_tyre = pd.concat(scenarios_tyre, ignore_index=True)

display(sim_tyre[[
    "Driver", "Event", "LapNumber", "Stint",
    "Compound", "TyreLife", "TrackTemp", "Rainfall",
    "predicted_laptime"
]])

,Driver,Event,LapNumber,Stint,Compound,TyreLife,TrackTemp,Rainfall,predicted_laptime
0,BOT,70th Anniversary Grand Prix,39,3,HARD,1,43.227586,False,110.701315
1,BOT,70th Anniversary Grand Prix,39,3,HARD,5,43.227586,False,91.438138
2,BOT,70th Anniversary Grand Prix,39,3,HARD,10,43.227586,False,91.438138
3,BOT,70th Anniversary Grand Prix,39,3,HARD,15,43.227586,False,91.438138
4,BOT,70th Anniversary Grand Prix,39,3,HARD,20,43.227586,False,91.412567
5,BOT,70th Anniversary Grand Prix,39,3,HARD,25,43.227586,False,91.398608


In [23]:
# Escenario 3: clima
weather_scenarios = [
    {"TrackTemp": 25.0, "Rainfall": False},
    {"TrackTemp": 35.0, "Rainfall": False},
    {"TrackTemp": 45.0, "Rainfall": False},
    {"TrackTemp": 25.0, "Rainfall": True},
    {"TrackTemp": 35.0, "Rainfall": True}
]

scenarios_weather = []

for w in weather_scenarios:
    row = base_row.copy()
    row["TrackTemp"] = w["TrackTemp"]
    row["Rainfall"] = w["Rainfall"]
    
    pred = model_sim.predict(row)[0]
    
    out = row.copy()
    out["scenario_type"] = "weather"
    out["scenario_value"] = f'TrackTemp={w["TrackTemp"]}, Rainfall={w["Rainfall"]}'
    out["predicted_laptime"] = pred
    scenarios_weather.append(out)

sim_weather = pd.concat(scenarios_weather, ignore_index=True)

display(sim_weather[[
    "Driver", "Event", "LapNumber", "Stint",
    "Compound", "TyreLife", "TrackTemp", "Rainfall",
    "predicted_laptime", "scenario_value"
]])

,Driver,Event,LapNumber,Stint,Compound,TyreLife,TrackTemp,Rainfall,predicted_laptime,scenario_value
0,BOT,70th Anniversary Grand Prix,39,3,HARD,7,25.0,False,95.456191,"TrackTemp=25.0, Rainfall=False"
1,BOT,70th Anniversary Grand Prix,39,3,HARD,7,35.0,False,90.765065,"TrackTemp=35.0, Rainfall=False"
2,BOT,70th Anniversary Grand Prix,39,3,HARD,7,45.0,False,91.438138,"TrackTemp=45.0, Rainfall=False"
3,BOT,70th Anniversary Grand Prix,39,3,HARD,7,25.0,True,97.185277,"TrackTemp=25.0, Rainfall=True"
4,BOT,70th Anniversary Grand Prix,39,3,HARD,7,35.0,True,92.494151,"TrackTemp=35.0, Rainfall=True"


In [24]:
realistic_scenarios = [
    {"Compound": "SOFT", "TyreLife": 2, "FreshTyre": True,  "Rainfall": False, "TrackTemp": 35.0},
    {"Compound": "MEDIUM", "TyreLife": 8, "FreshTyre": False, "Rainfall": False, "TrackTemp": 35.0},
    {"Compound": "HARD", "TyreLife": 15, "FreshTyre": False, "Rainfall": False, "TrackTemp": 35.0},
    {"Compound": "INTERMEDIATE", "TyreLife": 5, "FreshTyre": False, "Rainfall": True, "TrackTemp": 25.0},
    {"Compound": "WET", "TyreLife": 5, "FreshTyre": False, "Rainfall": True, "TrackTemp": 20.0}
]

scenarios_realistic = []

for s in realistic_scenarios:
    row = base_row.copy()
    for k, v in s.items():
        row[k] = v

    pred = model_sim.predict(row)[0]

    out = row.copy()
    out["scenario_type"] = "realistic_combined"
    out["scenario_value"] = str(s)
    out["predicted_laptime"] = pred
    scenarios_realistic.append(out)

sim_realistic = pd.concat(scenarios_realistic, ignore_index=True)

display(sim_realistic[[
    "Driver", "Event", "LapNumber", "Stint",
    "Compound", "TyreLife", "FreshTyre",
    "TrackTemp", "Rainfall", "predicted_laptime", "scenario_value"
]])

,Driver,Event,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackTemp,Rainfall,predicted_laptime,scenario_value
0,BOT,70th Anniversary Grand Prix,39,3,SOFT,2,True,35.0,False,98.607792,"{'Compound': 'SOFT', 'TyreLife': 2, 'FreshTyre..."
1,BOT,70th Anniversary Grand Prix,39,3,MEDIUM,8,False,35.0,False,90.862750,"{'Compound': 'MEDIUM', 'TyreLife': 8, 'FreshTy..."
2,BOT,70th Anniversary Grand Prix,39,3,HARD,15,False,35.0,False,90.862750,"{'Compound': 'HARD', 'TyreLife': 15, 'FreshTyr..."
3,BOT,70th Anniversary Grand Prix,39,3,INTERMEDIATE,5,False,25.0,True,98.676446,"{'Compound': 'INTERMEDIATE', 'TyreLife': 5, 'F..."
4,BOT,70th Anniversary Grand Prix,39,3,WET,5,False,20.0,True,98.510720,"{'Compound': 'WET', 'TyreLife': 5, 'FreshTyre'..."


In [25]:
import uuid

scenario_run_id = f"scenario_{uuid.uuid4().hex[:12]}"
print("scenario_run_id:", scenario_run_id)

sim_realistic_to_save = sim_realistic.copy()
sim_realistic_to_save["scenario_run_id"] = scenario_run_id

# Reordenar columnas si quieres
cols = [
    "scenario_run_id", "Year", "Event", "Driver", "Team", "LapNumber", "Stint",
    "Compound", "TyreLife", "FreshTyre", "TrackStatus",
    "AirTemp", "Humidity", "Pressure", "Rainfall",
    "TrackTemp", "WindDirection", "WindSpeed",
    "scenario_type", "scenario_value", "predicted_laptime"
]

sim_realistic_to_save = sim_realistic_to_save[cols]

sim_realistic_to_save.to_sql(
    "simulation_scenarios",
    engine,
    if_exists="append",
    index=False,
    method="multi"
)

print("Escenarios guardados en Supabase.")

scenario_run_id: scenario_23edc4f010b7
Escenarios guardados en Supabase.


In [26]:
query_final = """
SELECT *
FROM f1_simulation_base;
"""

df_final = pd.read_sql(query_final, engine)

print(df_final.shape)
display(df_final.head())
print(df_final.dtypes)

(145676, 34)


,Year,Event,Driver,DriverNumber,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,...,final_position,Points,Status,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,77,Mercedes,39,3,HARD,7.0,True,...,3.0,15.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
1,2020,70th Anniversary Grand Prix,ALB,23,Red Bull Racing,52,3,HARD,22.0,True,...,5.0,10.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
2,2020,70th Anniversary Grand Prix,ALB,23,Red Bull Racing,50,3,HARD,20.0,True,...,5.0,10.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
3,2020,70th Anniversary Grand Prix,ALB,23,Red Bull Racing,49,3,HARD,19.0,True,...,5.0,10.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
4,2020,70th Anniversary Grand Prix,ALB,23,Red Bull Racing,48,3,HARD,18.0,True,...,5.0,10.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


Year                        int64
Event                      object
Driver                     object
DriverNumber                int64
Team                       object
LapNumber                   int64
Stint                       int64
Compound                   object
TyreLife                  float64
FreshTyre                    bool
TrackStatus                object
LapTime           timedelta64[ns]
Sector1Time       timedelta64[ns]
Sector2Time       timedelta64[ns]
Sector3Time       timedelta64[ns]
SpeedI1                   float64
SpeedI2                   float64
SpeedFL                   float64
SpeedST                   float64
lap_position              float64
BroadcastName              object
Abbreviation               object
TeamName                   object
GridPosition              float64
final_position            float64
Points                    float64
Status                     object
AirTemp                   float64
Humidity                  float64
Pressure      

In [27]:
df_final = df_final[df_final["final_position"].notna()].copy()

print("Shape después de filtrar final_position:", df_final.shape)
print(df_final["final_position"].describe())

Shape después de filtrar final_position: (119070, 34)
count    119070.000000
mean          9.691240
std           5.389308
min           1.000000
25%           5.000000
50%          10.000000
75%          14.000000
max          20.000000
Name: final_position, dtype: float64


In [28]:
y_final = df_final["final_position"]

X_final = df_final[[
    "Year",
    "Event",
    "Driver",
    "Team",
    "LapNumber",
    "Stint",
    "Compound",
    "TyreLife",
    "FreshTyre",
    "TrackStatus",
    "GridPosition",
    "AirTemp",
    "Humidity",
    "Pressure",
    "Rainfall",
    "TrackTemp",
    "WindDirection",
    "WindSpeed"
]].copy()

print("X_final shape:", X_final.shape)
print("y_final shape:", y_final.shape)
display(X_final.head())

X_final shape: (119070, 18)
y_final shape: (119070,)


,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,GridPosition,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,1.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
1,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,52,3,HARD,22.0,True,1,9.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
2,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,50,3,HARD,20.0,True,1,9.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
3,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,49,3,HARD,19.0,True,1,9.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
4,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,48,3,HARD,18.0,True,1,9.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [33]:
from sklearn.model_selection import train_test_split

X_trainval_f, X_test_f, y_trainval_f, y_test_f = train_test_split(
    X_final, y_final,
    test_size=0.20,
    random_state=42
)

print("Train+Val:", X_trainval_f.shape, y_trainval_f.shape)
print("Test     :", X_test_f.shape, y_test_f.shape)

Train+Val: (95256, 18) (95256,)
Test     : (23814, 18) (23814,)


In [34]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

mccv_mae = []
mccv_rmse = []
mccv_r2 = []

n_repeats = 10

for i in range(n_repeats):
    X_train_f, X_val_f, y_train_f, y_val_f = train_test_split(
        X_trainval_f, y_trainval_f,
        test_size=0.30,
        random_state=42 + i
    )
    
    model_final.fit(X_train_f, y_train_f)
    y_val_pred_f = model_final.predict(X_val_f)
    
    mae = mean_absolute_error(y_val_f, y_val_pred_f)
    rmse = np.sqrt(mean_squared_error(y_val_f, y_val_pred_f))
    r2 = r2_score(y_val_f, y_val_pred_f)
    
    mccv_mae.append(mae)
    mccv_rmse.append(rmse)
    mccv_r2.append(r2)

print("MCCV - Monte Carlo Cross Validation")
print("MAE promedio :", np.mean(mccv_mae))
print("MAE std      :", np.std(mccv_mae))
print("RMSE promedio:", np.mean(mccv_rmse))
print("RMSE std     :", np.std(mccv_rmse))
print("R2 promedio  :", np.mean(mccv_r2))
print("R2 std       :", np.std(mccv_r2))

MCCV - Monte Carlo Cross Validation
MAE promedio : 0.6799212156173652
MAE std      : 0.042302824362728975
RMSE promedio: 1.2050102961352174
RMSE std     : 0.05181408448631819
R2 promedio  : 0.9499103038433298
R2 std       : 0.004452773325543431


In [37]:
model_final.fit(X_trainval_f, y_trainval_f)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Year', 'LapNumber', 'Stint',
                                                   'TyreLife', 'FreshTyre',
                                                   'GridPosition', 'AirTemp',
                                                   'Humidity', 'Pressure',
                                                   'Rainfall', 'TrackTemp',
                                                   'WindDirection',
                                                   'WindSpeed']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Event', 'Driver', 'Team',
                                                   'Compound',
                                                   'TrackStatus'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=15, n_estimators=30, n_jobs=-1,
                                       random_state=42))])

In [36]:
y_test_pred_f = model_final.predict(X_test_f)

mae_test_f = mean_absolute_error(y_test_f, y_test_pred_f)
rmse_test_f = np.sqrt(mean_squared_error(y_test_f, y_test_pred_f))
r2_test_f = r2_score(y_test_f, y_test_pred_f)

print("\nMétricas finales en TEST")
print("MAE test :", mae_test_f)
print("RMSE test:", rmse_test_f)
print("R2 test  :", r2_test_f)


Métricas finales en TEST
MAE test : 0.6397590654272614
RMSE test: 1.1361359714465982
R2 test  : 0.9554707419465676


In [38]:
import pandas as pd
import uuid
import numpy as np

run_id_laptime = f"laptime_{uuid.uuid4().hex[:10]}"

metrics_laptime = pd.DataFrame([{
    "run_id": run_id_laptime,
    "model_name": "RandomForestRegressor_realistic_v1",
    "target": "laptime_seconds",
    "train_rows": len(X_train),
    "val_rows": len(X_val),
    "test_rows": len(X_test),

    "mccv_mae_mean": float(np.mean(mccv_mae)),
    "mccv_rmse_mean": float(np.mean(mccv_rmse)),
    "mccv_r2_mean": float(np.mean(mccv_r2)),

    "val_mae": float(mae_val),
    "val_rmse": float(rmse_val),
    "val_r2": float(r2_val),

    "test_mae": float(mae_test),
    "test_rmse": float(rmse_test),
    "test_r2": float(r2_test),

    "notes": "Modelo realista sin sectores, sin velocidades y sin lap_position"
}])

display(metrics_laptime)

,run_id,model_name,target,train_rows,val_rows,test_rows,mccv_mae_mean,mccv_rmse_mean,mccv_r2_mean,val_mae,val_rmse,val_r2,test_mae,test_rmse,test_r2,notes
0,laptime_7653b18060,RandomForestRegressor_realistic_v1,laptime_seconds,79711,34162,28469,0.679921,1.20501,0.94991,1.835836,3.729013,0.986861,1.834544,3.746367,0.986096,"Modelo realista sin sectores, sin velocidades ..."


In [39]:
metrics_laptime.to_sql(
    "model_metrics",
    engine,
    if_exists="append",
    index=False,
    method="multi"
)

print("Métricas de LapTime guardadas.")

Métricas de LapTime guardadas.


In [42]:
run_id_final = f"finalpos_{uuid.uuid4().hex[:10]}"

metrics_final = pd.DataFrame([{
    "run_id": run_id_final,
    "model_name": "RandomForestRegressor_final_position_v1",
    "target": "final_position",
    "train_rows": len(X_train_f),
    "val_rows": len(X_val_f),
    "test_rows": len(X_test_f),

    "mccv_mae_mean": float(np.mean(mccv_mae)),
    "mccv_rmse_mean": float(np.mean(mccv_rmse)),
    "mccv_r2_mean": float(np.mean(mccv_r2)),


    "test_mae": float(mae_test_f),
    "test_rmse": float(rmse_test_f),
    "test_r2": float(r2_test_f),

    "notes": "Predicción de posición final a nivel de vuelta; MCCV solo en train"
}])

display(metrics_final)

,run_id,model_name,target,train_rows,val_rows,test_rows,mccv_mae_mean,mccv_rmse_mean,mccv_r2_mean,test_mae,test_rmse,test_r2,notes
0,finalpos_f35db52881,RandomForestRegressor_final_position_v1,final_position,66679,28577,23814,0.679921,1.20501,0.94991,0.639759,1.136136,0.955471,Predicción de posición final a nivel de vuelta...


In [43]:
metrics_final.to_sql(
    "model_metrics",
    engine,
    if_exists="append",
    index=False,
    method="multi"
)

print("Métricas de final_position guardadas.")

Métricas de final_position guardadas.


In [44]:
check_metrics = pd.read_sql("""
SELECT *
FROM model_metrics
ORDER BY created_at DESC
LIMIT 10;
""", engine)

display(check_metrics)

,id,run_id,model_name,target,created_at,train_rows,val_rows,test_rows,mccv_mae_mean,mccv_rmse_mean,mccv_r2_mean,val_mae,val_rmse,val_r2,test_mae,test_rmse,test_r2,notes
0,2,finalpos_f35db52881,RandomForestRegressor_final_position_v1,final_position,2026-03-07 23:02:28.768357,66679,28577,23814,0.679921,1.20501,0.94991,NaN,NaN,NaN,0.639759,1.136136,0.955471,Predicción de posición final a nivel de vuelta...
1,1,laptime_7653b18060,RandomForestRegressor_realistic_v1,laptime_seconds,2026-03-07 23:00:50.401292,79711,34162,28469,0.679921,1.20501,0.94991,1.835836,3.729013,0.986861,1.834544,3.746367,0.986096,"Modelo realista sin sectores, sin velocidades ..."


In [45]:
base_final = X_final.iloc[[0]].copy()

print("Fila base para final_position:")
display(base_final)

Fila base para final_position:


,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,GridPosition,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,1.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [46]:
base_final = X_final[
    (X_final["Compound"].isin(["SOFT", "MEDIUM", "HARD"])) &
    (X_final["Rainfall"] == False)
].iloc[[0]].copy()

display(base_final)

,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,GridPosition,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,1.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [47]:
grid_options = [1, 3, 5, 10, 15, 20]

scenarios_grid = []

for gp in grid_options:
    row = base_final.copy()
    row["GridPosition"] = gp

    pred = model_final.predict(row)[0]

    out = row.copy()
    out["scenario_type"] = "grid_position"
    out["scenario_value"] = gp
    out["predicted_final_position"] = pred
    scenarios_grid.append(out)

sim_grid = pd.concat(scenarios_grid, ignore_index=True)

display(sim_grid[[
    "Driver", "Event", "LapNumber", "Compound",
    "TyreLife", "GridPosition", "Rainfall",
    "predicted_final_position"
]])

,Driver,Event,LapNumber,Compound,TyreLife,GridPosition,Rainfall,predicted_final_position
0,BOT,70th Anniversary Grand Prix,39,HARD,7.0,1,False,2.134202
1,BOT,70th Anniversary Grand Prix,39,HARD,7.0,3,False,4.335040
2,BOT,70th Anniversary Grand Prix,39,HARD,7.0,5,False,4.528746
3,BOT,70th Anniversary Grand Prix,39,HARD,7.0,10,False,4.133333
4,BOT,70th Anniversary Grand Prix,39,HARD,7.0,15,False,5.400000
5,BOT,70th Anniversary Grand Prix,39,HARD,7.0,20,False,5.400000


In [49]:
grid_options = [1, 3, 5, 10, 15, 20]

scenarios_grid = []

for gp in grid_options:
    row = base_final.copy()
    row["GridPosition"] = gp

    pred = model_final.predict(row)[0]

    out = row.copy()
    out["scenario_type"] = "grid_position"
    out["scenario_value"] = gp
    out["predicted_final_position"] = pred
    scenarios_grid.append(out)

sim_grid = pd.concat(scenarios_grid, ignore_index=True)

display(sim_grid[[
    "Driver", "Event", "LapNumber", "Compound",
    "TyreLife", "GridPosition", "Rainfall",
    "predicted_final_position"
]])

,Driver,Event,LapNumber,Compound,TyreLife,GridPosition,Rainfall,predicted_final_position
0,BOT,70th Anniversary Grand Prix,39,HARD,7.0,1,False,2.134202
1,BOT,70th Anniversary Grand Prix,39,HARD,7.0,3,False,4.335040
2,BOT,70th Anniversary Grand Prix,39,HARD,7.0,5,False,4.528746
3,BOT,70th Anniversary Grand Prix,39,HARD,7.0,10,False,4.133333
4,BOT,70th Anniversary Grand Prix,39,HARD,7.0,15,False,5.400000
5,BOT,70th Anniversary Grand Prix,39,HARD,7.0,20,False,5.400000


In [50]:
compound_options = ["SOFT", "MEDIUM", "HARD"]

scenarios_compound_f = []

for comp in compound_options:
    row = base_final.copy()
    row["Compound"] = comp
    row["Rainfall"] = False

    pred = model_final.predict(row)[0]

    out = row.copy()
    out["scenario_type"] = "compound"
    out["scenario_value"] = comp
    out["predicted_final_position"] = pred
    scenarios_compound_f.append(out)

sim_compound_f = pd.concat(scenarios_compound_f, ignore_index=True)

display(sim_compound_f[[
    "Driver", "Event", "LapNumber", "GridPosition",
    "Compound", "TyreLife", "Rainfall",
    "predicted_final_position"
]])

,Driver,Event,LapNumber,GridPosition,Compound,TyreLife,Rainfall,predicted_final_position
0,BOT,70th Anniversary Grand Prix,39,1.0,SOFT,7.0,False,2.134202
1,BOT,70th Anniversary Grand Prix,39,1.0,MEDIUM,7.0,False,2.180929
2,BOT,70th Anniversary Grand Prix,39,1.0,HARD,7.0,False,2.134202


In [51]:
tyre_life_options = [1, 5, 10, 15, 20, 25]

scenarios_tyre_f = []

for tl in tyre_life_options:
    row = base_final.copy()
    row["TyreLife"] = tl
    row["Rainfall"] = False

    pred = model_final.predict(row)[0]

    out = row.copy()
    out["scenario_type"] = "tyre_life"
    out["scenario_value"] = tl
    out["predicted_final_position"] = pred
    scenarios_tyre_f.append(out)

sim_tyre_f = pd.concat(scenarios_tyre_f, ignore_index=True)

display(sim_tyre_f[[
    "Driver", "Event", "LapNumber", "GridPosition",
    "Compound", "TyreLife", "Rainfall",
    "predicted_final_position"
]])

,Driver,Event,LapNumber,GridPosition,Compound,TyreLife,Rainfall,predicted_final_position
0,BOT,70th Anniversary Grand Prix,39,1.0,HARD,1,False,2.134202
1,BOT,70th Anniversary Grand Prix,39,1.0,HARD,5,False,2.134202
2,BOT,70th Anniversary Grand Prix,39,1.0,HARD,10,False,2.134202
3,BOT,70th Anniversary Grand Prix,39,1.0,HARD,15,False,2.134202
4,BOT,70th Anniversary Grand Prix,39,1.0,HARD,20,False,2.134202
5,BOT,70th Anniversary Grand Prix,39,1.0,HARD,25,False,2.134202


In [52]:
realistic_final_scenarios = [
    {"GridPosition": 1,  "Compound": "SOFT",   "TyreLife": 2,  "FreshTyre": True,  "Rainfall": False, "TrackTemp": 35.0},
    {"GridPosition": 5,  "Compound": "MEDIUM", "TyreLife": 8,  "FreshTyre": False, "Rainfall": False, "TrackTemp": 35.0},
    {"GridPosition": 10, "Compound": "HARD",   "TyreLife": 15, "FreshTyre": False, "Rainfall": False, "TrackTemp": 35.0},
    {"GridPosition": 8,  "Compound": "INTERMEDIATE", "TyreLife": 5, "FreshTyre": False, "Rainfall": True, "TrackTemp": 25.0},
    {"GridPosition": 12, "Compound": "WET",    "TyreLife": 5,  "FreshTyre": False, "Rainfall": True, "TrackTemp": 20.0}
]

scenarios_realistic_f = []

for s in realistic_final_scenarios:
    row = base_final.copy()
    for k, v in s.items():
        row[k] = v

    pred = model_final.predict(row)[0]

    out = row.copy()
    out["scenario_type"] = "combined_realistic"
    out["scenario_value"] = str(s)
    out["predicted_final_position"] = pred
    scenarios_realistic_f.append(out)

sim_realistic_f = pd.concat(scenarios_realistic_f, ignore_index=True)

display(sim_realistic_f[[
    "Driver", "Event", "LapNumber", "GridPosition",
    "Compound", "TyreLife", "FreshTyre",
    "TrackTemp", "Rainfall", "predicted_final_position", "scenario_value"
]])

,Driver,Event,LapNumber,GridPosition,Compound,TyreLife,FreshTyre,TrackTemp,Rainfall,predicted_final_position,scenario_value
0,BOT,70th Anniversary Grand Prix,39,1,SOFT,2,True,35.0,False,2.134202,"{'GridPosition': 1, 'Compound': 'SOFT', 'TyreL..."
1,BOT,70th Anniversary Grand Prix,39,5,MEDIUM,8,False,35.0,False,4.604040,"{'GridPosition': 5, 'Compound': 'MEDIUM', 'Tyr..."
2,BOT,70th Anniversary Grand Prix,39,10,HARD,15,False,35.0,False,4.133333,"{'GridPosition': 10, 'Compound': 'HARD', 'Tyre..."
3,BOT,70th Anniversary Grand Prix,39,8,INTERMEDIATE,5,False,25.0,True,7.233333,"{'GridPosition': 8, 'Compound': 'INTERMEDIATE'..."
4,BOT,70th Anniversary Grand Prix,39,12,WET,5,False,20.0,True,4.300000,"{'GridPosition': 12, 'Compound': 'WET', 'TyreL..."


In [53]:
all_final_scenarios = pd.concat(
    [sim_grid, sim_compound_f, sim_tyre_f, sim_realistic_f],
    ignore_index=True
)

print(all_final_scenarios.shape)
display(all_final_scenarios.head(10))

(20, 21)


,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,...,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,scenario_type,scenario_value,predicted_final_position
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,grid_position,1,2.134202
1,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,grid_position,3,4.335040
2,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,grid_position,5,4.528746
3,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,grid_position,10,4.133333
4,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,grid_position,15,5.400000
5,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,grid_position,20,5.400000
6,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,SOFT,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,compound,SOFT,2.134202
7,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,MEDIUM,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,compound,MEDIUM,2.180929
8,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,compound,HARD,2.134202
9,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,1.0,True,1,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,tyre_life,1,2.134202


In [54]:
print(sorted(df["Team"].dropna().unique()))
print(sorted(df["Driver"].dropna().unique()))

['Alfa Romeo', 'Alfa Romeo Racing', 'AlphaTauri', 'Alpine', 'Aston Martin', 'Ferrari', 'Haas F1 Team', 'Kick Sauber', 'McLaren', 'Mercedes', 'RB', 'Racing Point', 'Red Bull Racing', 'Renault', 'Williams']
['AIT', 'ALB', 'ALO', 'BEA', 'BOT', 'COL', 'DEV', 'DOO', 'FIT', 'GAS', 'GIO', 'GRO', 'HAM', 'HUL', 'KUB', 'KVY', 'LAT', 'LAW', 'LEC', 'MAG', 'MAZ', 'MSC', 'NOR', 'OCO', 'PER', 'PIA', 'RAI', 'RIC', 'RUS', 'SAI', 'SAR', 'STR', 'TSU', 'VER', 'VET', 'ZHO']


In [55]:
base_combo = X_sim[
    (X_sim["Compound"].isin(["SOFT", "MEDIUM", "HARD"])) &
    (X_sim["Rainfall"] == False)
].iloc[[0]].copy()

display(base_combo)

,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7,True,1,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [56]:
combo_scenarios = [
    {"Driver": "VER", "Team": "Red Bull Racing"},
    {"Driver": "HAM", "Team": "Mercedes"},
    {"Driver": "LEC", "Team": "Ferrari"},
    {"Driver": "NOR", "Team": "McLaren"},
    {"Driver": "ALO", "Team": "Aston Martin"},
    {"Driver": "RUS", "Team": "Mercedes"},
    {"Driver": "SAI", "Team": "Ferrari"},
    {"Driver": "PIA", "Team": "McLaren"}
]

In [57]:
scenarios_combo = []

for s in combo_scenarios:
    row = base_combo.copy()
    row["Driver"] = s["Driver"]
    row["Team"] = s["Team"]
    row["Compound"] = "MEDIUM"
    row["TyreLife"] = 8
    row["FreshTyre"] = False
    row["Rainfall"] = False
    row["TrackTemp"] = 35.0

    pred = model_sim.predict(row)[0]

    out = row.copy()
    out["scenario_type"] = "driver_team_combo"
    out["scenario_value"] = f'{s["Driver"]} - {s["Team"]}'
    out["predicted_laptime"] = pred
    scenarios_combo.append(out)

sim_combo = pd.concat(scenarios_combo, ignore_index=True)

In [58]:
sim_combo_result = sim_combo[[
    "Driver", "Team", "Event", "LapNumber", "Stint",
    "Compound", "TyreLife", "TrackTemp", "Rainfall",
    "predicted_laptime"
]].sort_values("predicted_laptime").reset_index(drop=True)

sim_combo_result["ranking"] = sim_combo_result.index + 1

display(sim_combo_result[[
    "ranking", "Driver", "Team", "predicted_laptime",
    "Compound", "TyreLife", "TrackTemp", "Rainfall"
]])

,ranking,Driver,Team,predicted_laptime,Compound,TyreLife,TrackTemp,Rainfall
0,1,VER,Red Bull Racing,90.204810,MEDIUM,8,35.0,False
1,2,HAM,Mercedes,90.678447,MEDIUM,8,35.0,False
2,3,ALO,Aston Martin,90.911735,MEDIUM,8,35.0,False
3,4,LEC,Ferrari,90.945282,MEDIUM,8,35.0,False
4,5,NOR,McLaren,90.945282,MEDIUM,8,35.0,False
5,6,SAI,Ferrari,90.945282,MEDIUM,8,35.0,False
6,7,PIA,McLaren,90.945282,MEDIUM,8,35.0,False
7,8,RUS,Mercedes,91.117154,MEDIUM,8,35.0,False


In [59]:
import uuid

scenario_run_id = f"combo_laptime_{uuid.uuid4().hex[:10]}"
print("scenario_run_id:", scenario_run_id)

sim_combo_to_save = sim_combo.copy()
sim_combo_to_save = sim_combo_to_save.sort_values("predicted_laptime").reset_index(drop=True)
sim_combo_to_save["ranking"] = sim_combo_to_save.index + 1
sim_combo_to_save["scenario_run_id"] = scenario_run_id
sim_combo_to_save["notes"] = "Comparación piloto-equipo en escenario controlado de laptime"

cols = [
    "scenario_run_id",
    "Year", "Event", "Driver", "Team", "LapNumber", "Stint",
    "Compound", "TyreLife", "FreshTyre", "TrackStatus",
    "AirTemp", "Humidity", "Pressure", "Rainfall",
    "TrackTemp", "WindDirection", "WindSpeed",
    "scenario_type", "scenario_value", "predicted_laptime",
    "ranking", "notes"
]

sim_combo_to_save = sim_combo_to_save[cols]

display(sim_combo_to_save)

scenario_run_id: combo_laptime_728965b56f


,scenario_run_id,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,...,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,scenario_type,scenario_value,predicted_laptime,ranking,notes
0,combo_laptime_728965b56f,2020,70th Anniversary Grand Prix,VER,Red Bull Racing,39,3,MEDIUM,8,False,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,VER - Red Bull Racing,90.204810,1,Comparación piloto-equipo en escenario control...
1,combo_laptime_728965b56f,2020,70th Anniversary Grand Prix,HAM,Mercedes,39,3,MEDIUM,8,False,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,HAM - Mercedes,90.678447,2,Comparación piloto-equipo en escenario control...
2,combo_laptime_728965b56f,2020,70th Anniversary Grand Prix,ALO,Aston Martin,39,3,MEDIUM,8,False,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,ALO - Aston Martin,90.911735,3,Comparación piloto-equipo en escenario control...
3,combo_laptime_728965b56f,2020,70th Anniversary Grand Prix,LEC,Ferrari,39,3,MEDIUM,8,False,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,LEC - Ferrari,90.945282,4,Comparación piloto-equipo en escenario control...
4,combo_laptime_728965b56f,2020,70th Anniversary Grand Prix,NOR,McLaren,39,3,MEDIUM,8,False,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,NOR - McLaren,90.945282,5,Comparación piloto-equipo en escenario control...
5,combo_laptime_728965b56f,2020,70th Anniversary Grand Prix,SAI,Ferrari,39,3,MEDIUM,8,False,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,SAI - Ferrari,90.945282,6,Comparación piloto-equipo en escenario control...
6,combo_laptime_728965b56f,2020,70th Anniversary Grand Prix,PIA,McLaren,39,3,MEDIUM,8,False,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,PIA - McLaren,90.945282,7,Comparación piloto-equipo en escenario control...
7,combo_laptime_728965b56f,2020,70th Anniversary Grand Prix,RUS,Mercedes,39,3,MEDIUM,8,False,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,RUS - Mercedes,91.117154,8,Comparación piloto-equipo en escenario control...


In [60]:
sim_combo_to_save.to_sql(
    "simulation_driver_team_laptime",
    engine,
    if_exists="append",
    index=False,
    method="multi"
)

print("Simulación piloto-equipo guardada en Supabase.")

Simulación piloto-equipo guardada en Supabase.


In [61]:
check_combo = pd.read_sql("""
SELECT *
FROM simulation_driver_team_laptime
ORDER BY created_at DESC
LIMIT 20;
""", engine)

display(check_combo)

,id,scenario_run_id,created_at,Year,Event,Driver,Team,LapNumber,Stint,Compound,...,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,scenario_type,scenario_value,predicted_laptime,ranking,notes
0,1,combo_laptime_728965b56f,2026-03-07 23:20:34.696418,2020,70th Anniversary Grand Prix,VER,Red Bull Racing,39,3,MEDIUM,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,VER - Red Bull Racing,90.204810,1,Comparación piloto-equipo en escenario control...
1,2,combo_laptime_728965b56f,2026-03-07 23:20:34.696418,2020,70th Anniversary Grand Prix,HAM,Mercedes,39,3,MEDIUM,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,HAM - Mercedes,90.678447,2,Comparación piloto-equipo en escenario control...
2,3,combo_laptime_728965b56f,2026-03-07 23:20:34.696418,2020,70th Anniversary Grand Prix,ALO,Aston Martin,39,3,MEDIUM,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,ALO - Aston Martin,90.911735,3,Comparación piloto-equipo en escenario control...
3,4,combo_laptime_728965b56f,2026-03-07 23:20:34.696418,2020,70th Anniversary Grand Prix,LEC,Ferrari,39,3,MEDIUM,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,LEC - Ferrari,90.945282,4,Comparación piloto-equipo en escenario control...
4,5,combo_laptime_728965b56f,2026-03-07 23:20:34.696418,2020,70th Anniversary Grand Prix,NOR,McLaren,39,3,MEDIUM,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,NOR - McLaren,90.945282,5,Comparación piloto-equipo en escenario control...
5,6,combo_laptime_728965b56f,2026-03-07 23:20:34.696418,2020,70th Anniversary Grand Prix,SAI,Ferrari,39,3,MEDIUM,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,SAI - Ferrari,90.945282,6,Comparación piloto-equipo en escenario control...
6,7,combo_laptime_728965b56f,2026-03-07 23:20:34.696418,2020,70th Anniversary Grand Prix,PIA,McLaren,39,3,MEDIUM,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,PIA - McLaren,90.945282,7,Comparación piloto-equipo en escenario control...
7,8,combo_laptime_728965b56f,2026-03-07 23:20:34.696418,2020,70th Anniversary Grand Prix,RUS,Mercedes,39,3,MEDIUM,...,1000.950862,False,35.0,71.931034,2.236207,driver_team_combo,RUS - Mercedes,91.117154,8,Comparación piloto-equipo en escenario control...
